# 6. MLOps: Model Registry, Experiment Tracking, Feature Store & Monitoring

This notebook demonstrates the full MLOps infrastructure:
1. **Model Registry** — version, promote, A/B test ML models
2. **Experiment Tracker** — log runs, metrics, detect regressions
3. **Feature Store** — compute, cache, version, and trace features
4. **Production Monitor** — embedding drift, confidence calibration, LLM health
5. **Evaluation Pipeline** — golden-set benchmarks for all components

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import json
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## 6.1 Model Registry

In [ ]:
from server.mlops.model_registry import ModelRegistry, ModelStage

registry = ModelRegistry()
print(registry.summary())

In [ ]:
# Register a new version of the hypothesis grounder
registry.register(
    name='hypothesis_grounder',
    version='1.1.0',
    model_type='contrastive',
    framework='numpy',
    metrics={'brier_score': 0.08, 'calibration_error': 0.05},
    hyperparameters={'d_embed': 128, 'temperature': 0.05},
    description='v1.1: doubled embedding dimension, lower temperature',
)

# List versions
for v in registry.list_versions('hypothesis_grounder'):
    print(f'  {v.version}  stage={v.stage.value}  metrics={v.metrics}')

In [ ]:
# Set up A/B test: route 20% traffic to v1.1.0
ab = registry.set_ab_test('hypothesis_grounder', '1.0.0', '1.1.0', split=0.2)

# Simulate routing
from collections import Counter
routes = Counter(registry.route('hypothesis_grounder') for _ in range(1000))
print(f'A/B test routing (1000 requests): {dict(routes)}')

# Record metrics for each version
for _ in range(50):
    registry.record_ab_metric('hypothesis_grounder', '1.0.0', 'confidence', np.random.normal(0.72, 0.1))
    registry.record_ab_metric('hypothesis_grounder', '1.1.0', 'confidence', np.random.normal(0.78, 0.08))

ab_results = registry.get_ab_results('hypothesis_grounder')
print(f'\nA/B Results:')
print(f'  Control (v1.0.0):   n={ab_results["control_n"]}')
print(f'  Treatment (v1.1.0): n={ab_results["treatment_n"]}')

In [ ]:
# Promote v1.1.0 to production
registry.promote('hypothesis_grounder', '1.1.0', ModelStage.PRODUCTION)
active = registry.get_active('hypothesis_grounder')
print(f'Active model: {active.name}:{active.version} (stage={active.stage.value})')

---
## 6.2 Experiment Tracker

In [ ]:
from server.mlops.experiment_tracker import ExperimentTracker

tracker = ExperimentTracker()

# Simulate 30 hypothesis agent runs
for i in range(30):
    run = tracker.start_run(
        experiment_name='hypothesis_agent',
        model_name='hypothesis_grounder',
        model_version='1.0.0',
        session_id=i + 1,
    )
    run.log_param('temperature', 0.07)
    run.log_param('d_embed', 64)
    run.log_input(f'CO2 reduction on Cu(111) query {i}')
    run.log_metric('confidence', 0.6 + np.random.normal(0, 0.1))
    run.log_metric('intermediate_f1', 0.7 + np.random.normal(0, 0.08))
    run.log_tokens(input_tokens=1200 + np.random.randint(-200, 200),
                   output_tokens=400 + np.random.randint(-100, 100))
    run.log_output(f'hypothesis output for query {i}')
    tracker.end_run(run.run_id)

print(tracker.summary())

In [ ]:
# Metric history and latency
conf_history = tracker.metric_history('hypothesis_agent', 'confidence')
f1_history = tracker.metric_history('hypothesis_agent', 'intermediate_f1')
latency = tracker.latency_percentiles('hypothesis_agent')
cost = tracker.cost_summary(since_hours=24)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(conf_history, 'o-', markersize=4, color='steelblue')
axes[0].set_title('Confidence over runs'); axes[0].set_ylabel('Confidence')

axes[1].plot(f1_history, 's-', markersize=4, color='green')
axes[1].set_title('Intermediate F1 over runs'); axes[1].set_ylabel('F1')

axes[2].bar(['p50', 'p90', 'p99'], [latency['p50'], latency['p90'], latency['p99']], color='coral')
axes[2].set_title('Latency percentiles'); axes[2].set_ylabel('ms')

plt.tight_layout()
plt.show()

print(f'Daily cost: ${sum(cost.values()):.4f}')

# Check for regression
reg = tracker.detect_regression('hypothesis_agent', 'confidence', window=10)
print(f'Regression check: {reg or "No regression detected"}')


---
## 6.3 Feature Store

In [ ]:
from server.feature_store.store import FeatureStore

store = FeatureStore()
print(store.summary())

In [ ]:
# Compute mechanism graph features from a reaction network
co2rr_network = json.dumps({
    'reaction_network': [
        {'lhs': ['CO2(g)', '*', 'H+', 'e-'], 'rhs': ['COOH*']},
        {'lhs': ['COOH*', 'H+', 'e-'], 'rhs': ['CO*', 'H2O(g)']},
    ],
    'intermediates': ['*', 'CO2(g)', 'COOH*', 'CO*', 'CO(g)', 'H2O(g)'],
    'ts_edges': [['CO2*', 'COOH*'], ['COOH*', 'CO*']],
    'coads_pairs': [['CO*', 'H*']],
})

features = store.compute('mechanism_graph', 'co2rr_cu111', co2rr_network)
print(f'Mechanism features: {features}')
print(f'Shape: {features.shape}')
print(f'Feature names: [n_inter, n_steps, n_ts, n_coads, n_surf, n_gas, n_ec, frac_ec]')

In [ ]:
# Compute thermo features
thermo_data = json.dumps({
    'dG_profile': [0.0, 0.22, -0.15, -0.45, -1.10],
    'overpotential_V': 0.61,
    'U_limiting_V': -0.72,
})
thermo_feats = store.compute('thermo_profile', 'co2rr_cu111_thermo', thermo_data)
print(f'Thermo features: {thermo_feats}')
print(f'Feature names: [n_steps, min_dG, max_dG, rds_barrier, rds_idx, eta, U_lim]')

In [ ]:
# Trace provenance
prov = store.trace_provenance('co2rr_cu111')
print(f'\nProvenance for co2rr_cu111:')
print(json.dumps(prov, indent=2, default=str))

In [ ]:
# Lineage tracking
lineage = store.get_lineage('co2rr_cu111')
print(f'Lineage records: {len(lineage)}')
for l in lineage:
    print(f'  feature={l.feature_name}, input_hash={l.input_hash}, version={l.model_version}')

---
## 6.4 Production Monitoring

In [ ]:
from server.mlops.monitoring import ProductionMonitor

monitor = ProductionMonitor()

# Simulate embedding observations
rng = np.random.default_rng(42)
for i in range(200):
    vec = rng.normal(0, 1, 1536).tolist()
    monitor.embedding.record(vec)

# Simulate confidence observations
for _ in range(100):
    conf = np.clip(rng.normal(0.75, 0.15), 0, 1)
    correct = rng.random() < conf  # well-calibrated
    monitor.calibration.record(conf, correct)

# Simulate RAG hits
for _ in range(100):
    hit = rng.random() < 0.7
    rank = rng.integers(1, 6) if hit else 0
    monitor.retrieval.record(hit, rank)

# Simulate LLM calls
for _ in range(50):
    success = rng.random() < 0.97
    latency = int(rng.exponential(2000))
    json_ok = rng.random() < 0.95
    monitor.llm.record(success, latency, json_ok)
    monitor.cost.record(rng.exponential(0.01))

# Check all monitors
health = monitor.health_status()
print(f'System healthy: {health["healthy"]}')
print(f'Calibration ECE: {health["calibration_ece"]:.3f}')
print(f'LLM success rate: {health["llm_success_rate"]:.1%}')
print(f'Daily cost: ${health["daily_cost_usd"]:.2f}')
print(f'Embedding drift: {health["embedding_drift"]}')

alerts = monitor.check_all()
if alerts:
    print(f'\nAlerts ({len(alerts)}):')
    for a in alerts:
        print(f'  {a}')
else:
    print('\nNo alerts — all systems nominal.')

---
## 6.5 Evaluation Pipeline — Golden-Set Benchmarks

In [ ]:
from science.evaluation.metrics import (
    IntentParsingMetrics, HypothesisMetrics, ThermodynamicsMetrics,
    RAGMetrics, GrounderMetrics, SCFPredictionMetrics, GOLDEN_SET,
)

print(f'Golden set: {len(GOLDEN_SET)} benchmark examples\n')
for ex in GOLDEN_SET:
    print(f'  {ex.id}: "{ex.query[:60]}..."')
    print(f'    Expected intermediates: {ex.expected_intermediates}')
    print(f'    Expected overpotential: {ex.expected_overpotential} V')
    print()

In [ ]:
# Run individual metric evaluations

# 1. Intent parsing
intent_results = IntentParsingMetrics.evaluate(
    predicted={'stage': 'electrocatalysis', 'system': {'material': 'Cu', 'facet': '111'}},
    expected={'stage': 'electrocatalysis', 'system': {'material': 'Cu', 'facet': '111'}},
)
print('Intent metrics:')
for r in intent_results:
    print(f'  {r.name}: {r.value}')

# 2. Hypothesis
hyp_results = HypothesisMetrics.evaluate(
    predicted_intermediates=['*', 'CO2(g)', 'COOH*', 'CO*', 'extra_species*'],
    expected_intermediates=['*', 'CO2(g)', 'COOH*', 'CO*', 'CO(g)', 'H2O(g)'],
)
print('\nHypothesis metrics:')
for r in hyp_results:
    print(f'  {r.name}: {r.value:.3f}')

# 3. Thermodynamics
thermo_results = ThermodynamicsMetrics.evaluate(
    predicted_dG=[0.0, 0.25, -0.10, -0.50, -1.05],
    expected_dG=[0.0, 0.22, -0.15, -0.45, -1.10],
    predicted_eta=0.65, expected_eta=0.61,
)
print('\nThermodynamics metrics:')
for r in thermo_results:
    print(f'  {r.name}: {r.value:.4f}')

# 4. RAG
print(f'\nRAG MRR examples:')
print(f'  [T, F, F] → MRR = {RAGMetrics.mrr([True, False, False])}')
print(f'  [F, T, F] → MRR = {RAGMetrics.mrr([False, True, False])}')
print(f'  [F, F, T] → MRR = {RAGMetrics.mrr([False, False, True])}')
print(f'  NDCG@5 example: {RAGMetrics.ndcg_at_k([3, 2, 1, 0, 0], k=5):.3f}')

In [ ]:
# Grounder calibration analysis
confidences = np.random.beta(5, 2, 200).tolist()
labels = [int(np.random.random() < c) for c in confidences]

brier = GrounderMetrics.brier_score(confidences, labels)
ece = GrounderMetrics.calibration_error(confidences, labels)
print(f'Grounder calibration:')
print(f'  Brier score: {brier:.4f} (0=perfect, 1=worst)')
print(f'  ECE: {ece:.4f} (0=perfectly calibrated)')

# Calibration plot
fig, ax = plt.subplots(figsize=(5, 5))
n_bins = 10
c_arr = np.array(confidences)
y_arr = np.array(labels)
for i in range(n_bins):
    lo, hi = i / n_bins, (i + 1) / n_bins
    mask = (c_arr >= lo) & (c_arr < hi)
    if mask.sum() > 0:
        ax.bar((lo + hi) / 2, y_arr[mask].mean(), width=1/n_bins * 0.8,
               color='steelblue', alpha=0.7)
ax.plot([0, 1], [0, 1], 'r--', label='Perfect calibration')
ax.set_xlabel('Predicted confidence'); ax.set_ylabel('Observed accuracy')
ax.set_title(f'Reliability diagram (ECE={ece:.3f})')
ax.legend()
plt.tight_layout()
plt.show()